Notebook 3 – Data Quality Assessment
=

Objective: 
=
Identify missing values, duplicate records, invalid values, incorrect data types,
and other data quality problems before performing analysis or building machine learning models.
    
Import libraries and create a sample dataset
=

In [ ]:
import pandas as pd
import numpy as np

data = {
    "Customer_ID": [101, 102, 103, 104, 104, 106, 107, 108],
    "Name": ["Arun", "Bala", "Charan", "Divya", "Divya", "Ezhil", "", "Gopi"],
    "Age": [25, 30, np.nan, -5, -5, 150, 28, 35],
    "Gender": ["Male", "Female", "Male", "Female",
               "Female", "Other", "Unknown", "Male"],
    "Salary": [25000, 35000, 40000, 30000, 30000,
               -1000, np.nan, 45000],
    "Join_Date": ["2025-01-10", "2025-02-15", "2025-03-20",
                  "2025-04-10", "2025-04-10", "invalid",
                  "2025-06-15", "2025-07-20"]
}

df = pd.DataFrame(data)

print(df)

Explanation: 
=
This dataset intentionally contains missing values, duplicate records, negative salaries, invalid ages, unexpected categories, empty strings, and invalid dates.
These issues help us practise data quality assessment

Missing values
=
    
Number of missing values
=

This returns the number of missing (NaN or None) values in each column.    

In [ ]:
print(df.isnull().sum())

Total missing values
=

In [ ]:
total_missing = df.isnull().sum().sum()
print("Total missing values:", total_missing)

Percentage of missing values
=

In [ ]:
missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

print(missing_percentage)

Missing value patterns
=

In [ ]:
print(df.isnull())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.imshow(df.isnull(), aspect="auto", cmap="viridis")
plt.xticks(range(len(df.columns)), df.columns, rotation=45)
plt.yticks(range(len(df)))
plt.title("Missing Value Pattern")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.colorbar(label="Missing: 1, Present: 0")
plt.tight_layout()
plt.show()

Interpretation: 
=

Identify which columns contain missing values and whether multiple missing values occur in the same rows. Decide whether to remove, replace, or investigate those values based on their meaning.
Duplicate records

Duplicate records
=

Identify duplicate rows

Displays rows that duplicate earlier rows across all columns.

In [ ]:
print(df[df.duplicated()])

Count duplicate rows
=

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Identify completely duplicated rows
=

In [ ]:
complete_duplicates = df[
    df.duplicated(keep=False)
]

print(complete_duplicates)

Interpretation:
=

Duplicate records can distort customer counts, averages, and model training. Investigate them before deleting anything, because repeated IDs or 
names are not always errors.

Invalid values
=
Negative values where inappropriate
=



In [ ]:
negative_age = df[df["Age"] < 0]
negative_salary = df[df["Salary"] < 0]

print("Negative ages:")
print(negative_age)

print("Negative salaries:")
print(negative_salary)

Interpretation:
=
Negative age is generally invalid. A negative salary may be invalid or may represent a refund or accounting adjustment, depending on the dataset.

Impossible values
=
invalid_age = df[    (df["Age"] < 0) | (df["Age"] > 120)]print(invalid_age)



The age limit of 120 is an illustrative validation rule; adjust it for the dataset and business context.
    
Incorrect categories
=    

valid_genders = ["Male", "Female", "Other"]invalid_categories = df[    ~df["Gender"].isin(valid_genders)]print(invalid_categories)



Here, "Unknown" is outside the permitted category list. Depending on the data dictionary, it may be better classified as missing or unknown rather than
automatically deleted.
    
Invalid dates
=

df["Join_Date_Parsed"] = pd.to_datetime(    df["Join_Date"],    errors="coerce")invalid_dates = df[    df["Join_Date_Parsed"].isna()]print(invalid_dates[["Customer_ID", "Join_Date"]])



errors="coerce" converts unparseable dates to NaT, making them easier to identify.

Incorrect data types
=

print(df.dtypes)



Convert columns to their intended types when appropriate:

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")print(df.dtypes)



Interpretation: Incorrect types can cause calculation errors, failed date comparisons, or incorrect analysis.

Unexpected values
=
print("Gender values:")print(df["Gender"].unique())print("Age values:")print(df["Age"].unique())print("Salary values:")print(df["Salary"].unique())



Use a documented list of valid values or business rules to identify unexpected values. A value being unusual does not automatically make it invalid.

Empty strings
=
empty_strings = df.apply(    lambda col: col.astype("string").str.strip().eq("").sum())print("Empty strings by column:")print(empty_strings)




Null-like values
=
Values such as "NA", "N/A", "null", "None", and "unknown" can appear as text instead of actual missing values.
null_like_values = ["NA", "N/A", "null", "None", "unknown"]for col in df.select_dtypes(include=["object", "string"]).columns:    df[col] = df[col].replace(        r"^\s*(?:NA|N/A|null|None|unknown)\s*$",        np.nan,        regex=True,        case=False    )print(df.isnull().sum())



This code standardizes the listed text values as missing. Confirm first that those words do not have legitimate meanings in your dataset.

Create a data quality summary
=


Important:

This notebook identifies data quality issues; it should not automatically remove or change suspicious values without checking the dataset's rules.

In [ ]:
quality_report = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().mean() * 100
    ).round(2),
    "Unique_Values": df.nunique(dropna=True),
    "Duplicate_Values": [
        df[col].duplicated().sum()
        for col in df.columns
    ]
})

print(quality_report)

In [ ]:
quality_report.to_csv(
    "data_quality_report.csv",
    index=True,
    index_label="Column"
)

Notebook 4 – Univariate Analysis
=

Objective: 
=

Analyze one variable at a time to understand its distribution, central tendency, variability, outliers, and frequency patterns.
    
Import libraries and prepare the dataset
=
You can continue with the previous dataset or load your cleaned dataset.






In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use the existing df, or load a dataset:
# df = pd.read_csv("cleaned_dataset.csv")

print(df.head())
print(df.shape)
print(df.dtypes)

In [ ]:
age = pd.to_numeric(df["Age"], errors="coerce")
age = age[age.between(0, 120)].dropna()

salary = pd.to_numeric(df["Salary"], errors="coerce")
salary = salary[salary >= 0].dropna()

These filters are illustrative. Use domain-specific validation rules for your real dataset


Numerical variables
=
We will use Age and Salary to demonstrate descriptive statistics.
    
Mean, median, and mode

- Mean: Average value.
    
- Median: Middle value after sorting.
    
- Mode: Most frequently occurring value.
    
Interpretation: If the mean salary is much higher than the median, a few high salaries may be pulling the average upward.

In [ ]:
print("Mode Age:", age.mode().tolist())

print("Mean Salary:", salary.mean())
print("Median Salary:", salary.median())
print("Mode Salary:", salary.mode().tolist())

Minimum, maximum, and range
=
Range is the difference between the maximum and minimum values

In [ ]:
print("Minimum Age:", age.min())
print("Maximum Age:", age.max())
print("Age Range:", age.max() - age.min())

print("Minimum Salary:", salary.min())
print("Maximum Salary:", salary.max())
print("Salary Range:", salary.max() - salary.min())

Variance and standard deviation
=
Pandas calculates sample variance and sample standard deviation by default (ddof=1).
    
- Variance: Measures squared deviations from the mean.

- Standard deviation: Measures spread in the original units.
    
Interactive visual explanation:

In [ ]:
print("Age Variance:", age.var())
print("Age Standard Deviation:", age.std())

print("Salary Variance:", salary.var())
print("Salary Standard Deviation:", salary.std())

Quartiles and percentiles
=

25th percentile: 25% of observations are at or below this value.



50th percentile: Median.



75th percentile: 75% of observations are at or below this value.



90th percentile: 90% of observations are at or below this value

In [ ]:
print("Salary Quartiles:")
print(salary.quantile([0.25, 0.50, 0.75]))

print("Salary 90th Percentile:", salary.quantile(0.90))

Skewness and kurtosis
=

Interpretation:
=
- Skewness near zero suggests approximate symmetry.
    
- Positive skewness indicates a longer right tail.
    
- Negative skewness indicates a longer left tail.
    
- Pandas kurtosis reports excess kurtosis; values near zero indicate kurtosis similar to a normal distribution under this measure.
    
These are descriptive indicators, not definitive tests of normality.

In [ ]:
print("Salary Skewness:", salary.skew())
print("Salary Kurtosis:", salary.kurt())

Numerical visualizations
=

Histogram
=    
A histogram shows how numerical values are distributed across intervals.
plt.figure(figsize=(8, 5))plt.hist(age, bins=10, edgecolor="black")plt.title("Age Distribution")plt.xlabel("Age")plt.ylabel("Frequency")plt.tight_layout()plt.show()



Findings to explain: Identify the most common age range, whether the distribution is symmetric or skewed, and whether there are unusually high or low values.

KDE / Density plot
=
A KDE plot provides a smoothed estimate of the distribution.
age.plot(    kind="kde",    figsize=(8, 5),    title="Age Density Plot")plt.xlabel("Age")plt.ylabel("Density")plt.tight_layout()plt.show()



Findings to explain: Peaks indicate areas where observations are concentrated. Multiple peaks may indicate different groups within the dataset.
    
Box plot
=    
A box plot summarizes the median, quartiles, and potential outliers.
plt.figure(figsize=(8, 4))plt.boxplot(age, vert=False)plt.title("Age Box Plot")plt.xlabel("Age")plt.tight_layout()plt.show()



Create a box plot for salary too:
=    
plt.figure(figsize=(8, 4))plt.boxplot(salary, vert=False)plt.title("Salary Box Plot")plt.xlabel("Salary")plt.tight_layout()plt.show()



Findings to explain: Check the median, spread, asymmetry, and potential outliers. A plotted outlier is not necessarily an error; investigate it before removing it.

Categorical variables
=

Categorical variables represent groups, labels, or classes, such as gender, department, product category, or payment method.
Frequency and unique values

gender = df["Gender"].dropna()print("Unique genders:")print(gender.unique())print("Number of unique genders:")print(gender.nunique())print("Frequency:")print(gender.value_counts())



Value counts and percentage distribution
=

frequency = gender.value_counts()percentage = (    gender.value_counts(normalize=True) * 100).round(2)category_summary = pd.DataFrame({    "Frequency": frequency,    "Percentage": percentage})print(category_summary)



Interpretation: Frequency tells us how many records belong to each category. Percentage shows each category's share of the non-missing observations.
Remember to validate categories such as "Unknown" before treating them as genuine categories.

Bar chart
=
frequency.plot(    kind="bar",    figsize=(8, 5),    edgecolor="black")plt.title("Gender Distribution")plt.xlabel("Gender")plt.ylabel("Count")plt.xticks(rotation=0)plt.tight_layout()plt.show()



Count plot
=
A count plot is a categorical chart that displays the number of observations in each category. With Matplotlib, the following produces the same basic result:
plt.figure(figsize=(8, 5))plt.bar(frequency.index.astype(str), frequency.values)plt.title("Gender Count Plot")plt.xlabel("Gender")plt.ylabel("Count")plt.tight_layout()plt.show()



Pie chart
=
Use a pie chart when there are only a few categories and the proportions are easy to compare.
frequency.plot(    kind="pie",    autopct="%1.1f%%",    figsize=(7, 7))plt.title("Gender Percentage Distribution")plt.ylabel("")plt.tight_layout()plt.show()



For many categories, a bar chart is usually easier to interpret.

Categorical variables
=
Categorical variables represent groups, labels, or classes, such as gender, department, product category, or payment method.
    
4.1 Frequency and unique values

In [ ]:
gender = df["Gender"].dropna()

print("Unique genders:")
print(gender.unique())

print("Number of unique genders:")
print(gender.nunique())

print("Frequency:")
print(gender.value_counts())

Value counts and percentage distribution
=

In [ ]:
frequency = gender.value_counts()

percentage = (
    gender.value_counts(normalize=True) * 100
).round(2)

category_summary = pd.DataFrame({
    "Frequency": frequency,
    "Percentage": percentage
})

print(category_summary)

Interpretation: 
=
Frequency tells us how many records belong to each category. Percentage shows each category's share of the non-missing observations.
Remember to validate categories such as "Unknown" before treating them as genuine categories.

Bar chart
=

In [ ]:
frequency.plot(
    kind="bar",
    figsize=(8, 5),
    edgecolor="black"
)

plt.title("Gender Distribution")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Count plot
=
A count plot is a categorical chart that displays the number of observations in each category. With Matplotlib, 
the following produces the same basic result:

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(frequency.index.astype(str), frequency.values)

plt.title("Gender Count Plot")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Pie chart
=
Use a pie chart when there are only a few categories and the proportions are easy to compare.

In [ ]:
frequency.plot(
    kind="pie",
    autopct="%1.1f%%",
    figsize=(7, 7)
)

plt.title("Gender Percentage Distribution")
plt.ylabel("")
plt.tight_layout()
plt.show()

Analyze every important variable automatically
=
The following code generates descriptive statistics for all numerical columns and frequency summaries for all categorical columns.

In [ ]:
# Numerical summary
numerical_columns = df.select_dtypes(
    include="number"
).columns

print("NUMERICAL ANALYSIS")

for col in numerical_columns:
    values = df[col].dropna()

    if values.empty:
        print(f"\n{col}: No valid numerical values")
        continue

    print(f"\n--- {col} ---")
    print("Count:", values.count())
    print("Mean:", values.mean())
    print("Median:", values.median())
    print("Mode:", values.mode().tolist())
    print("Minimum:", values.min())
    print("Maximum:", values.max())
    print("Range:", values.max() - values.min())
    print("Variance:", values.var())
    print("Standard Deviation:", values.std())
    print("Quartiles:")
    print(values.quantile([0.25, 0.50, 0.75]))
    print("Skewness:", values.skew())
    print("Kurtosis:", values.kurt())

Save the analysis results
=
Save numerical descriptive statistics

In [ ]:
numerical_summary = df.select_dtypes(
    include="number"
).describe().T

numerical_summary.to_csv(
    "univariate_numerical_summary.csv"
)

print("Numerical summary saved successfully.")

In [ ]:
with pd.ExcelWriter("univariate_categorical_summary.xlsx") as writer:
    for col in categorical_columns:
        summary = pd.DataFrame({
            "Frequency": df[col].value_counts(dropna=False),
            "Percentage": (
                df[col].value_counts(
                    normalize=True,
                    dropna=False
                ) * 100
            ).round(2)
        })

        summary.to_excel(
            writer,
            sheet_name=col[:31] or "Category"
        )
